# WidowX End Effector Pose Monitor

This notebook listens to the live Interbotix ROS 2 interface and prints the current end effector pose.

Before running the notebook, start the robot driver in another terminal. Example:

```bash
ros2 launch interbotix_xsarm_control xsarm_control.launch robot_model:=wx250
```

Then run the cells below. The monitor prints:

- position: `x`, `y`, `z`
- orientation: `roll`, `pitch`, `yaw` in radians and degrees
- the full `T_sb` transform matrix


In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

from datetime import datetime
from math import degrees

from IPython.display import clear_output
import interbotix_common_modules.angle_manipulation as ang
from interbotix_common_modules.common_robot.robot import robot_shutdown
from interbotix_xs_modules.xs_robot.arm import InterbotixManipulatorXS
import numpy as np
import rclpy

ROBOT_MODEL = 'wx250s'
ROBOT_NAME = None
GROUP_NAME = 'arm'
GRIPPER_NAME = 'gripper'
UPDATE_HZ = 5.0

np.set_printoptions(precision=4, suppress=True)


In [ ]:
def read_ee_pose(bot):
    T_sb = bot.arm.get_ee_pose()
    x, y, z = T_sb[:3, 3]
    roll, pitch, yaw = ang.rotation_matrix_to_euler_angles(T_sb[:3, :3])
    return {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'x': float(x),
        'y': float(y),
        'z': float(z),
        'roll': float(roll),
        'pitch': float(pitch),
        'yaw': float(yaw),
        'T_sb': T_sb,
    }


def print_ee_pose(pose):
    print(f"timestamp: {pose['timestamp']}")
    print()
    print('Position [m]')
    print(f"  x: {pose['x']:+0.4f}")
    print(f"  y: {pose['y']:+0.4f}")
    print(f"  z: {pose['z']:+0.4f}")
    print()
    print('Orientation [rad]')
    print(f"  roll : {pose['roll']:+0.4f}")
    print(f"  pitch: {pose['pitch']:+0.4f}")
    print(f"  yaw  : {pose['yaw']:+0.4f}")
    print()
    print('Orientation [deg]')
    print(f"  roll : {degrees(pose['roll']):+7.2f}")
    print(f"  pitch: {degrees(pose['pitch']):+7.2f}")
    print(f"  yaw  : {degrees(pose['yaw']):+7.2f}")
    print()
    print('T_sb')
    print(pose['T_sb'])


In [ ]:
if 'bot' not in globals() or bot is None:
    bot = InterbotixManipulatorXS(
        robot_model=ROBOT_MODEL,
        robot_name=ROBOT_NAME,
        group_name=GROUP_NAME,
        gripper_name=GRIPPER_NAME,
    )

print(f'Connected to {ROBOT_MODEL}.')
print('Run the monitor cell below. Use Kernel Interrupt to stop it.')


In [ ]:
refresh_period = 1.0 / UPDATE_HZ

try:
    while True:
        rclpy.spin_once(bot.core.get_node(), timeout_sec=refresh_period)
        pose = read_ee_pose(bot)
        clear_output(wait=True)
        print(f'Listening to {ROBOT_MODEL} at {UPDATE_HZ:.1f} Hz')
        print('Press the notebook stop button or interrupt the kernel to stop monitoring.')
        print()
        print_ee_pose(pose)
except KeyboardInterrupt:
    print('Pose monitoring stopped.')


In [ ]:
# Optional cleanup cell.
# Run this if you are finished with the notebook and want to close the ROS node cleanly.

if 'bot' in globals() and bot is not None:
    robot_shutdown(bot.core.get_node())
    bot = None
    print('Robot node shut down.')
else:
    print('No active robot connection found.')
